# Reproduce revised core results

This notebook reproduces the revised core results used in the manuscript and reviewer response:

- repeated-run feasibility estimates for Table 3,
- Section X confidence-interval tables,
- residual-buffer ablation,
- UM|BO p75/p90/p95 sensitivity analysis.

Figures 3--5 in the manuscript are retained as relaxed_oracle diagnostic figures. This notebook focuses on the additional repeated-run analyses introduced during revision.


# Setup and data paths

In [ ]:
# ==========================
# Cell 1 — Imports
# ==========================
# If running in Google Colab, uncomment:
try:
except Exception:
    pass

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, ParameterGrid
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# CatBoost is optional
try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False

print("CatBoost available:", CATBOOST_AVAILABLE)

from pathlib import Path


In [ ]:
# Repository-relative paths.
# Run this notebook either from the repository root or from the notebooks/ directory.

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data" / "processed" / "engineered_metadata.csv"

TABLE_OUTDIR = REPO_ROOT / "outputs" / "tables"
FIGURE_OUTDIR = REPO_ROOT / "outputs" / "figures"
REVIEW_OUTDIR = REPO_ROOT / "outputs" / "review_validation"

for _p in [TABLE_OUTDIR, FIGURE_OUTDIR, REVIEW_OUTDIR]:
    _p.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_PATH

print("Repository root:", REPO_ROOT)
print("Input data:", CSV_PATH)
print("Table outputs:", TABLE_OUTDIR)
print("Figure outputs:", FIGURE_OUTDIR)
print("Review outputs:", REVIEW_OUTDIR)

# Data loading and grouped preprocessing

In [ ]:
# ==========================
# Cell 3 — Load + Prepare Data (df0)
# ==========================

df_raw = pd.read_csv(CSV_PATH)
print("Raw:", df_raw.shape)

# required columns check
need_cols = ["uid", GROUP_COL, TARGET]
for c in need_cols:
    assert c in df_raw.columns, f"Missing required column: {c}"
for c in FEATURES:
    assert c in df_raw.columns, f"Missing feature column: {c}"

# keep discharge only if column exists
df0 = df_raw.copy()
if "type" in df0.columns:
    df0 = df0[df0["type"].astype(str).str.lower().eq("discharge")].copy()

# coerce numeric
df0[GROUP_COL] = df0[GROUP_COL].astype(str)
df0["uid"] = pd.to_numeric(df0["uid"], errors="coerce")
df0[TARGET] = pd.to_numeric(df0[TARGET], errors="coerce")
for f in FEATURES:
    df0[f] = pd.to_numeric(df0[f], errors="coerce")

# drop missing
df0 = df0.dropna(subset=[GROUP_COL, "uid", TARGET] + FEATURES).copy()

# sort (important for any time-aware logic later)
df0 = df0.sort_values([GROUP_COL, "uid"]).reset_index(drop=True)

print("Prepared df0:", df0.shape, "| batteries:", df0[GROUP_COL].nunique())
print("Target mean:", round(df0[TARGET].mean(), 3), "| std:", round(df0[TARGET].std(), 3))

In [ ]:
# ==========================
# Cell 4 — Group-safe split (battery-level) + optional IQR cleaning
# ==========================
def add_age_fraction(df, group_col=GROUP_COL, order_col="uid"):
    """
    Adds _age_frac in [0,1] based on within-battery rank of order_col.
    Assumes df is already sorted by [group_col, order_col] (you do this).
    """
    out = df.copy()
    # rank starting at 0
    out["_age_rank"] = out.groupby(group_col)[order_col].rank(method="first") - 1
    out["_age_n"]    = out.groupby(group_col)[order_col].transform("count")
    out["_age_frac"] = out["_age_rank"] / np.maximum(1.0, (out["_age_n"] - 1.0))
    return out


def make_split(
    df0,
    split_seed,
    use_iqr=USE_IQR_CLEANING,
    protocol="standard",
    early_frac=SHIFT_EARLY_FRAC,
    late_frac=SHIFT_LATE_FRAC,
    min_cycles=SHIFT_MIN_CYCLES,
    order_col=SHIFT_ORDER_COL
):
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_BATTERY_FRAC, random_state=split_seed)
    tr_idx, te_idx = next(gss.split(df0, groups=df0[GROUP_COL]))

    df_train = df0.iloc[tr_idx].copy().reset_index(drop=True)
    df_test  = df0.iloc[te_idx].copy().reset_index(drop=True)

    # leakage check
    assert set(df_train[GROUP_COL]).isdisjoint(set(df_test[GROUP_COL])), "Leakage: shared batteries!"

    # ---- aging-shift filter (train early, test late) ----
    if protocol == "aging_shift":
        df_train = add_age_fraction(df_train, group_col=GROUP_COL, order_col=order_col)
        df_test  = add_age_fraction(df_test,  group_col=GROUP_COL, order_col=order_col)

        # optionally remove very short batteries (stabilizes the filter)
        df_train = df_train[df_train["_age_n"] >= min_cycles].copy()
        df_test  = df_test[df_test["_age_n"] >= min_cycles].copy()

        # early part for training, late part for testing
        df_train = df_train[df_train["_age_frac"] <= early_frac].copy()
        df_test  = df_test[df_test["_age_frac"] >= (1.0 - late_frac)].copy()

        # drop helper cols
        df_train = df_train.drop(columns=["_age_rank","_age_n","_age_frac"])
        df_test  = df_test.drop(columns=["_age_rank","_age_n","_age_frac"])

    # ---- optional IQR cleaning (fit bounds on *filtered* train) ----
    if use_iqr:
        q1 = df_train[TARGET].quantile(0.25)
        q3 = df_train[TARGET].quantile(0.75)
        iqr = max(0.0, q3 - q1)

        lb = max(0.0, q1 - 1.5 * iqr)
        ub = max(lb, q3 + 1.5 * iqr)   # ensure ub >= lb



        before_tr, before_te = len(df_train), len(df_test)
        df_train = df_train[df_train[TARGET].between(lb, ub)].copy().reset_index(drop=True)
        if IQR_APPLY_TO_TEST:
            df_test = df_test[df_test[TARGET].between(lb, ub)].copy().reset_index(drop=True)

        print(f"[{protocol} | split {split_seed}] IQR bounds: [{lb:.3f}, {ub:.3f}] | "
              f"train {len(df_train)}/{before_tr} | test {len(df_test)}/{before_te}")

    return df_train.reset_index(drop=True), df_test.reset_index(drop=True)


# quick test run (light sanity)
df_train, df_test = make_split(df0, split_seed=CFG["split_seeds"][0], use_iqr=USE_IQR_CLEANING)
print("Train batteries:", df_train[GROUP_COL].nunique(), "| Test batteries:", df_test[GROUP_COL].nunique())
print("Train rows:", len(df_train), "| Test rows:", len(df_test))

# Forecasting model, residual buffer, policies, and metrics

In [ ]:
# ==========================
# Cell 5 — Model fit (group-safe) + OOF residuals (train only)
# Supports: RandomForest ("rf") + optional CatBoost ("cat")
# ==========================

def fit_predict(df_train, df_test, model_name="rf", model_seed=42):
    X_train = df_train[FEATURES].to_numpy()
    y_train = df_train[TARGET].to_numpy().astype(float)
    g_train = df_train[GROUP_COL].to_numpy()

    X_test  = df_test[FEATURES].to_numpy()
    y_test  = df_test[TARGET].to_numpy().astype(float)

    # group-safe CV
    n_groups = len(np.unique(g_train))
    n_splits = min(N_GROUP_CV_SPLITS, n_groups)
    gkf = GroupKFold(n_splits=n_splits)

    # ----------------------
    # Model: RandomForest
    # ----------------------
    if model_name == "rf":
        grid = RF_GRIDS[CFG["rf_grid"]]
        best_mae = None
        best_params = None

        for params in ParameterGrid(grid):
            maes = []
            for tr, va in gkf.split(X_train, y_train, g_train):
                m = RandomForestRegressor(random_state=model_seed, n_jobs=-1, **params)
                m.fit(X_train[tr], y_train[tr])
                pred = m.predict(X_train[va])
                maes.append(mean_absolute_error(y_train[va], pred))
            avg = float(np.mean(maes))
            if best_mae is None or avg < best_mae:
                best_mae = avg
                best_params = params

        # OOF
        oof = np.zeros_like(y_train, dtype=float)
        for tr, va in gkf.split(X_train, y_train, g_train):
            m = RandomForestRegressor(random_state=model_seed, n_jobs=-1, **best_params)
            m.fit(X_train[tr], y_train[tr])
            oof[va] = m.predict(X_train[va])

        resid = np.abs(y_train - oof)

        # final fit + test pred
        final = RandomForestRegressor(random_state=model_seed, n_jobs=-1, **best_params)
        final.fit(X_train, y_train)
        y_hat = final.predict(X_test).astype(float)

        return {
            "model": "rf",
            "best_params": best_params,
            "cv_mae": float(best_mae),
            "y_train": y_train,
            "oof": oof,
            "resid": resid,
            "y_true": y_test,
            "y_hat": y_hat,
        }

    # ----------------------
    # Model: CatBoost (optional)
    # ----------------------
    if model_name == "cat":
        assert CATBOOST_AVAILABLE, "CatBoost not installed/available in this environment."

        # Simple grid (keep small first; expand in heavy mode later)
        # CatBoost handles nonlinearity well; we keep it light and stable.
        cat_grid = [
            {"depth": 6, "learning_rate": 0.1, "iterations": 1000},
            {"depth": 8, "learning_rate": 0.05, "iterations": 2000},
        ]

        best_mae = None
        best_params = None

        for params in cat_grid:
            maes = []
            for tr, va in gkf.split(X_train, y_train, g_train):
                m = CatBoostRegressor(
                    random_seed=model_seed,
                    loss_function="MAE",
                    verbose=False,
                    **params
                )
                m.fit(X_train[tr], y_train[tr])
                pred = m.predict(X_train[va])
                maes.append(mean_absolute_error(y_train[va], pred))
            avg = float(np.mean(maes))
            if best_mae is None or avg < best_mae:
                best_mae = avg
                best_params = params

        # OOF
        oof = np.zeros_like(y_train, dtype=float)
        for tr, va in gkf.split(X_train, y_train, g_train):
            m = CatBoostRegressor(
                random_seed=model_seed,
                loss_function="MAE",
                verbose=False,
                **best_params
            )
            m.fit(X_train[tr], y_train[tr])
            oof[va] = m.predict(X_train[va])

        resid = np.abs(y_train - oof)

        # final fit + test pred
        final = CatBoostRegressor(
            random_seed=model_seed,
            loss_function="MAE",
            verbose=False,
            **best_params
        )
        final.fit(X_train, y_train)
        y_hat = final.predict(X_test)

        return {
            "model": "cat",
            "best_params": best_params,
            "cv_mae": float(best_mae),
            "y_train": y_train,
            "oof": oof,
            "resid": resid,
            "y_true": y_test,
            "y_hat": np.array(y_hat, dtype=float),
        }

    raise ValueError("model_name must be 'rf' or 'cat'")


# quick test run on current split
art = fit_predict(df_train, df_test, model_name="rf", model_seed=CFG["model_seeds"][0])
print("Model:", art["model"], "| CV MAE:", round(art["cv_mae"], 3), "| resid q80:", round(np.quantile(art["resid"], 0.8), 3))
print("Test MAE:", round(mean_absolute_error(art["y_true"], art["y_hat"]), 3))

In [ ]:
# ==========================
# Cell 6B — Controllers + Metrics (same logic as old notebook)
# ==========================

from collections import deque
import numpy as np
from collections import deque


def rolling_conformal_q(err, alpha=0.1, W=200, q_init=None, smooth=0.9):
    """
    err = (y_hat - y_true). Positive means overestimation.
    Returns q_t such that y_lower = y_hat - q_t is conservative.
    Uses only past errors.
    """
    if q_init is None:
        q_init = np.quantile(err, 1 - alpha)

    buf = deque(maxlen=W)
    q = q_init
    q_series = np.zeros_like(err, dtype=float)

    for t in range(len(err)):
        if len(buf) >= max(20, W//5):
            q_raw = float(np.quantile(np.array(buf), 1 - alpha))
            q = smooth*q + (1 - smooth)*q_raw
        q_series[t] = q
        buf.append(float(err[t]))

    return q_series


def time_weighted_floor(floors, split):
    return (split['active']*floors['active'] +
            split['quiet'] *floors['quiet']  +
            split['predawn']*floors['predawn'])


def ml_pct_from_forecast(y_hat, T, safety, qbuf, floors, split):
    # conservative runtime (forecast minus buffer)
    safe_runtime = np.maximum(0.0, y_hat - qbuf)
    pct_raw = 100.0 * safety * (safe_runtime / T)
    floor_avg = time_weighted_floor(floors, split)
    return np.clip(pct_raw, floor_avg, 100.0)


def apply_hysteresis_and_ramp(pct, hysteresis=HYSTERESIS, ramp=RAMP_LIMIT):
    pct = np.asarray(pct, dtype=float).copy()
    if len(pct) == 0:
        return pct

    out = np.zeros_like(pct)
    out[0] = pct[0]

    for i in range(1, len(pct)):
        desired = pct[i]
        prev = out[i-1]

        # hysteresis deadband
        if abs(desired - prev) <= hysteresis:
            out[i] = prev
            continue

        # ramp limit
        delta = desired - prev
        delta = np.clip(delta, -ramp, ramp)
        out[i] = prev + delta

    return out


def avg_to_phase_arrays(pct_avg, floors, split, priority=PRIORITY):
    """
    Convert avg brightness pct_avg into per-phase arrays (active/quiet/predawn),
    keeping floors, and allocating any remaining budget by priority.
    """
    pct_avg = np.asarray(pct_avg, dtype=float)
    n = len(pct_avg)

    # floors per phase
    pct_phase = {k: np.full(n, float(floors[k]), dtype=float) for k in floors}
    floor_avg = time_weighted_floor(floors, split)

    rem = np.maximum(0.0, pct_avg - floor_avg)

    # allocate remainder to the first phase in priority order
    for k in priority:
        if split[k] <= 0:
            continue
        add = rem / split[k]
        pct_phase[k] += add
        rem = np.zeros_like(rem)
        break

    pct_eff = np.zeros(n, dtype=float)
    for k in floors:
        pct_eff += pct_phase[k] * float(split[k])

    detail = {"floor_avg": float(floor_avg)}
    return pct_phase["active"], pct_phase["quiet"], pct_phase["predawn"], pct_eff, detail

def simulate_metrics_phase(phase_arrays, T, y_true, split, phase_order=None):
    """
    Phase-aware evaluator.

    phase_arrays : dict
        Example:
        {
            "active":  np.array([...], dtype=float),
            "quiet":   np.array([...], dtype=float),
            "predawn": np.array([...], dtype=float),
        }

    T : float
        Total horizon duration in seconds.

    y_true : array-like
        True runtime at 100% brightness, in seconds.

    split : dict
        Phase fractions over the total horizon.

    phase_order : list or None
        Temporal order of phases. If None, uses ["active", "quiet", "predawn"]
        filtered by availability.

    Returns:
        metrics dict with same keys as your old evaluator.
    """
    y_true = np.asarray(y_true, dtype=float)
    n = len(y_true)

    if phase_order is None:
        phase_order = [p for p in ["active", "quiet", "predawn"] if p in phase_arrays]

    unmet_minutes = np.zeros(n, dtype=float)
    blackout = np.zeros(n, dtype=bool)
    commanded_demand = np.zeros(n, dtype=float)  # in "100%-seconds"

    for t in range(n):
        remaining_budget_true = float(y_true[t])

        elapsed_real_time = 0.0
        cumulative_demand = 0.0
        depleted = False

        # total commanded demand over full schedule (used for ES, same semantics as before)
        total_demand_t = 0.0

        for p in phase_order:
            d_p = float(T) * float(split[p])               # real seconds in this phase
            b_p = float(np.clip(phase_arrays[p][t], 0.0, 100.0))  # brightness %
            demand_p = d_p * (b_p / 100.0)               # 100%-seconds consumed by this phase

            total_demand_t += demand_p

            if depleted:
                elapsed_real_time += d_p
                continue

            # If this phase does not exhaust the battery
            if cumulative_demand + demand_p <= remaining_budget_true + 1e-12:
                cumulative_demand += demand_p
                elapsed_real_time += d_p
                continue

            # Depletion occurs inside this phase
            blackout[t] = True
            depleted = True

            if b_p <= 1e-12:
                # zero-brightness edge case; no extra demand in this phase
                depletion_time_within_phase = d_p
            else:
                remaining_budget_for_phase = max(0.0, remaining_budget_true - cumulative_demand)
                depletion_time_within_phase = remaining_budget_for_phase / (b_p / 100.0)
                depletion_time_within_phase = np.clip(depletion_time_within_phase, 0.0, d_p)

            depletion_clock = elapsed_real_time + depletion_time_within_phase
            unmet_sec = max(0.0, T - depletion_clock)
            unmet_minutes[t] = unmet_sec / 60.0

            # after depletion, the rest of the horizon is unmet
            elapsed_real_time = T
            cumulative_demand = remaining_budget_true

        commanded_demand[t] = total_demand_t

    blackout_rate = 100.0 * blackout.mean()

    if blackout.any():
        um_bo = float(np.mean(unmet_minutes[blackout]))
        um_bo_p90 = float(np.percentile(unmet_minutes[blackout], 90))
    else:
        um_bo = 0.0
        um_bo_p90 = 0.0

    # Keep ES semantics aligned with the rest of your paper:
    # compare commanded demand to full-brightness all-night demand (= T)
    energy_saving = 100.0 * np.mean(1.0 - (commanded_demand / float(T)))

    return {
        "blackout_rate": blackout_rate,
        "energy_saving": energy_saving,
        "unmet_minutes_blackout": um_bo,
        "unmet_minutes_blackout_p90": um_bo_p90,
        "unmet_minutes_all_mean": float(np.mean(unmet_minutes)),
        "blackout_mask": blackout,
        "unmet_minutes_per_night": unmet_minutes,
    }

def simulate_metrics(pct, T, y_true):
    """
    Matches KTP_Dimming_Controller_Pipeline.ipynb

    Assumption:
      y_true = runtime at 100% brightness
      if dim to pct, achievable runtime increases ~ (100/pct)

    Outputs:
      blackout_rate in PERCENT (0..100)
      energy_saving in PERCENT (0..100)
    """
    y_true = np.asarray(y_true, dtype=float)
    pct = np.asarray(pct, dtype=float)

    pct = np.clip(pct, 1.0, 100.0)
    achievable = y_true * (100.0 / pct)

    blackout = achievable < T
    blackout_rate = float(blackout.mean() * 100.0)
    energy_saving = float(100.0 - pct.mean())

    unmet = np.maximum(0.0, T - achievable) / 60.0
    um_all = float(unmet.mean())
    um_blackout = float(unmet[blackout].mean()) if blackout.any() else 0.0
    um_p90_blackout = float(np.quantile(unmet[blackout], 0.90)) if blackout.any() else 0.0

    return dict(
        blackout_rate=blackout_rate,
        energy_saving=energy_saving,
        unmet_minutes=um_all,
        unmet_minutes_blackout=um_blackout,
        unmet_minutes_blackout_p90=um_p90_blackout,
    )




def br_at_constant_avg(pct_avg, T, y_true):
    """
    Baseline blackout-rate under constant avg brightness pct_avg.
    """
    pct_avg = float(pct_avg)
    y_true = np.asarray(y_true, dtype=float)
    # keep consistent with your older usage: blackout depends on y_true vs T only
    return float(np.mean(y_true < T))


def schedule_baseline_avg_brightness(schedule, floors, split):
    """
    Returns avg brightness implied by schedule dict across phases.
    """
    bA = float(schedule["active"])
    bQ = float(schedule["quiet"])
    bP = float(schedule["predawn"])
    bavg = split["active"]*bA + split["quiet"]*bQ + split["predawn"]*bP
    return bA, bQ, bP, float(bavg)


# ---- Rule baseline (voltage-score → {100,70,40}%)
def rule_setpoints(df_sub, hysteresis=HYSTERESIS, ramp=RAMP_LIMIT):
    sv, ev = df_sub['start_voltage'].values, df_sub['end_voltage'].values
    svn = (sv - sv.min()) / (sv.max() - sv.min() + 1e-9)
    evn = (ev - ev.min()) / (ev.max() - ev.min() + 1e-9)
    score = 0.7*svn + 0.3*evn

    def bucket(v):
        if v >= 0.66: return 100.0
        if v >= 0.33: return 70.0
        return 40.0

    pct = np.array([bucket(v) for v in score], float)
    return apply_hysteresis_and_ramp(pct, hysteresis, ramp)


# ---- Fixed-safety ML: choose safety s from grid to meet blackout budget
def choose_fixed_safety(y_true, y_hat, T, qbuf, floors, split, budget, s_grid=S_GRID):
    best_fallback = None

    # fallback tracking
    for s in s_grid:
        pct = apply_hysteresis_and_ramp(
            ml_pct_from_forecast(y_hat, T, s, qbuf, floors, split)
        )

        _, _, _, pct_eff, _ = avg_to_phase_arrays(pct, floors, split, PRIORITY)

        m = simulate_metrics_from_pct_eff(
            pct_eff=pct_eff,
            T=T,
            y_true=y_true,
            floors=floors,
            split=split,
            priority=PRIORITY
        )

        if (best_fallback is None or
            m['blackout_rate'] < best_fallback[2]['blackout_rate'] - 1e-12 or
            (abs(m['blackout_rate'] - best_fallback[2]['blackout_rate']) <= 1e-12 and
             m['unmet_minutes_all_mean'] < best_fallback[2]['unmet_minutes_all_mean'])):
            best_fallback = (s, pct_eff, m)

    # pick BRIGHTEST within budget (reverse scan)
    for s in s_grid[::-1]:
        pct = apply_hysteresis_and_ramp(
            ml_pct_from_forecast(y_hat, T, s, qbuf, floors, split)
        )

        _, _, _, pct_eff, _ = avg_to_phase_arrays(pct, floors, split, PRIORITY)

        m = simulate_metrics_from_pct_eff(
            pct_eff=pct_eff,
            T=T,
            y_true=y_true,
            floors=floors,
            split=split,
            priority=PRIORITY
        )

        if m['blackout_rate'] <= budget:
            return (s, pct_eff, m)

    return best_fallback


# ---- Block-fixed ML: re-select s per block of B cycles
def block_fixed_controller(y_true, y_hat, T, qbuf, floors, split, budget, B=30):
    n = len(y_true)
    pct_out = np.zeros(n, float)

    for i in range(0, n, B):
        j = min(n, i+B)
        q_block = qbuf if np.isscalar(qbuf) else qbuf[i:j]
        s, pct_block, _ = choose_fixed_safety(
            y_true[i:j], y_hat[i:j], T, q_block, floors, split, budget
        )
        pct_out[i:j] = pct_block

    pct_out = apply_hysteresis_and_ramp(pct_out)

    m = simulate_metrics_from_pct_eff(
        pct_eff=pct_out,
        T=T,
        y_true=y_true,
        floors=floors,
        split=split,
        priority=PRIORITY
    )

    return pct_out, m


# ---- Adaptive ML: proportional update to track blackout-rate budget (POLICY-AWARE)
def _smooth_step(prev, desired, hysteresis, ramp_limit):
    if abs(desired - prev) <= hysteresis:
        return prev
    delta = np.clip(desired - prev, -ramp_limit, ramp_limit)
    return prev + delta

def solve_weighted_phase_allocation(remaining_budget_sec, T, floors, split, weights):
    """
    Solve a lightweight phase-allocation problem for one horizon segment.

    remaining_budget_sec : safe runtime budget in '100%-seconds'
    T                    : remaining horizon duration in seconds
    floors               : dict of phase floors
    split                : dict of phase fractions over the CURRENT remaining horizon
    weights              : dict of phase importance weights

    Returns:
        b_phase : dict with brightness command for each phase (in %)
    """
    phases = [p for p in split.keys() if split[p] > 0]

    # phase durations in seconds over the current remaining horizon
    d = {p: float(T) * float(split[p]) for p in phases}

    # start from floors
    b = {p: float(floors[p]) for p in phases}

    # floor budget
    floor_cost = sum(d[p] * (b[p] / 100.0) for p in phases)

    # if even floors exceed the remaining safe budget, keep floors
    # this matches your strict comfort-floor interpretation
    if remaining_budget_sec <= floor_cost + 1e-12:
        return b

    # extra budget beyond floors
    extra = remaining_budget_sec - floor_cost

    # allocate extra budget by descending phase priority / weight
    order = sorted(phases, key=lambda p: (-weights[p], p))

    for p in order:
        if d[p] <= 0:
            continue

        max_extra_cost = d[p] * ((100.0 - b[p]) / 100.0)
        use = min(extra, max_extra_cost)

        b[p] += 100.0 * use / d[p]
        extra -= use

        if extra <= 1e-12:
            break

    return b


def renormalize_remaining_split(phases_left, original_split):
    """
    Renormalize split fractions for the remaining phases so they sum to 1.
    """
    total = sum(float(original_split[p]) for p in phases_left)
    if total <= 0:
        return {p: 0.0 for p in phases_left}
    return {p: float(original_split[p]) / total for p in phases_left}


def mpc_phase_controller(
    y_true, y_hat, T, qbuf, floors, split, weights,
    rho=1.0,
    smooth_output=False,
    hysteresis=HYSTERESIS, ramp_limit=RAMP_LIMIT
):
    """
    Phase-level MPC-style baseline.

    rho : scalar tightening factor applied to the safe runtime budget.
          Smaller rho => more conservative controller.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_hat  = np.asarray(y_hat,  dtype=float)
    n = len(y_true)

    if np.isscalar(qbuf):
        q_arr = np.full(n, float(qbuf), dtype=float)
    else:
        q_arr = np.asarray(qbuf, dtype=float)

    phase_order = [p for p in ["active", "quiet", "predawn"] if split[p] > 0]
    schedule_debug = []

    for t in range(n):
        safe_budget = float(rho) * max(0.0, float(y_hat[t] - q_arr[t]))  # <-- budget-aware
        remaining_budget = safe_budget

        phase_cmds = {}

        for k, phase_now in enumerate(phase_order):
            phases_left = phase_order[k:]
            split_left = renormalize_remaining_split(phases_left, split)

            T_left = float(T) * sum(float(split[p]) for p in phases_left)

            alloc = solve_weighted_phase_allocation(
                remaining_budget_sec=remaining_budget,
                T=T_left,
                floors={p: floors[p] for p in phases_left},
                split=split_left,
                weights={p: weights[p] for p in phases_left}
            )

            b_now = float(np.clip(alloc[phase_now], floors[phase_now], 100.0))
            phase_cmds[phase_now] = b_now

            d_now = float(T) * float(split[phase_now])
            remaining_budget = max(0.0, remaining_budget - d_now * (b_now / 100.0))

        schedule_debug.append(phase_cmds)

    phase_arrays = {
        p: np.array([sched[p] for sched in schedule_debug], dtype=float)
        for p in phase_order
    }

    # Keep smoothing OFF for the main comparator; otherwise the optimized phase
    # schedule gets flattened and reprojected.
    if smooth_output:
        for p in phase_order:
            phase_arrays[p] = apply_hysteresis_and_ramp(
                phase_arrays[p],
                hysteresis=hysteresis,
                ramp=ramp_limit
            )

    pct_eff = np.zeros(n, dtype=float)
    for p in phase_order:
        pct_eff += float(split[p]) * phase_arrays[p]

    metrics = simulate_metrics_phase(
        phase_arrays=phase_arrays,
        T=T,
        y_true=y_true,
        split=split,
        phase_order=["active", "quiet", "predawn"]
    )

    debug = {
        "phase_schedules": schedule_debug,
        "phase_arrays": phase_arrays,
        "pct_eff": pct_eff.copy(),
        "rho": float(rho),
    }

    return pct_eff, metrics, debug

def simulate_metrics_from_pct_eff(pct_eff, T, y_true, floors, split, priority):
    """
    Convert a single effective average brightness per night into
    phase commands using the existing allocator, then evaluate phase-aware.
    """
    b_active, b_quiet, b_predawn, _, _ = avg_to_phase_arrays(
        pct_eff, floors, split, priority
    )

    phase_arrays = {
        "active":  np.asarray(b_active, dtype=float),
        "quiet":   np.asarray(b_quiet, dtype=float),
        "predawn": np.asarray(b_predawn, dtype=float),
    }

    return simulate_metrics_phase(
        phase_arrays=phase_arrays,
        T=T,
        y_true=y_true,
        split=split,
        phase_order=["active", "quiet", "predawn"]
    )
def choose_mpc_reserve(
    y_true, y_hat, T, qbuf, floors, split, weights, budget,
    rho_grid=np.linspace(0.50, 1.00, 11)
):
    """
    Select the LARGEST rho that satisfies the blackout-rate budget on the
    training-side data. Falls back to the lowest-BR candidate if none satisfy.
    """
    best_fallback = None

    for rho in rho_grid:
        pct_eff, m, _ = mpc_phase_controller(
            y_true=y_true,
            y_hat=y_hat,
            T=T,
            qbuf=qbuf,
            floors=floors,
            split=split,
            weights=weights,
            rho=float(rho),
            smooth_output=False
        )

        if (best_fallback is None or
            m["blackout_rate"] < best_fallback[2]["blackout_rate"] - 1e-12 or
            (abs(m["blackout_rate"] - best_fallback[2]["blackout_rate"]) <= 1e-12 and
             m["unmet_minutes_all_mean"] < best_fallback[2]["unmet_minutes_all_mean"])):
            best_fallback = (float(rho), pct_eff, m)

    # pick the brightest / least conservative rho that still meets the budget
    for rho in rho_grid[::-1]:
        pct_eff, m, _ = mpc_phase_controller(
            y_true=y_true,
            y_hat=y_hat,
            T=T,
            qbuf=qbuf,
            floors=floors,
            split=split,
            weights=weights,
            rho=float(rho),
            smooth_output=False
        )

        if m["blackout_rate"] <= budget:
            return float(rho), pct_eff, m

    return best_fallback
def simulate_literal_phase_schedule(schedule, T, y_true, split):
    """
    Evaluate an exact fixed phase schedule, e.g. 100/50/70, without collapsing
    it to an average brightness first.
    """
    y_true = np.asarray(y_true, dtype=float)
    n = len(y_true)

    phase_arrays = {
        "active":  np.full(n, float(schedule["active"]),  dtype=float),
        "quiet":   np.full(n, float(schedule["quiet"]),   dtype=float),
        "predawn": np.full(n, float(schedule["predawn"]), dtype=float),
    }

    return simulate_metrics_phase(
        phase_arrays=phase_arrays,
        T=T,
        y_true=y_true,
        split=split,
        phase_order=["active", "quiet", "predawn"]
    )
def adaptive_controller_exp(
    y_true, y_hat, T, qbuf, floors, split, budget,
    window=30,
    kappa=0.02,
    s_init=1.0,
    hysteresis=5.0,
    ramp_limit=10.0,
    s_min=0.1,
    s_max=5.0
):
    """
    Closed-loop adaptive safety factor (exponential update) using DELIVERED brightness.

    Update:
        s <- s * exp(kappa * (budget - BR_recent_pct))
    where budget and BR_recent_pct are in percent.

    With your mapping, higher s => higher brightness => higher blackout risk,
    so this sign is correct (reduces s when BR is too high).
    """
    y_true = np.asarray(y_true, float)
    y_hat  = np.asarray(y_hat, float)
    n = len(y_true)

    s = float(s_init)

    pct_cmd = np.zeros(n, float)
    pct_del = np.zeros(n, float)
    s_hist  = np.zeros(n, float)
    br_hist = np.zeros(n, float)

    for t in range(n):
        # commanded brightness from forecast
        pct_t = float(ml_pct_from_forecast(
            np.array([y_hat[t]]), T, s, qbuf, floors, split
        )[0])
        pct_t = float(np.clip(pct_t, 1.0, 100.0))
        pct_cmd[t] = pct_t

        # actuator constraints ONLINE
        if t == 0:
            pct_del[t] = pct_cmd[t]
        else:
            pct_del[t] = _smooth_step(pct_del[t-1], pct_cmd[t], hysteresis, ramp_limit)

        # recent BR based on DELIVERED brightness
        start = max(0, t - window + 1)
        pct_win = np.clip(pct_del[start:t+1], 1.0, 100.0)
        ach_win = y_true[start:t+1] * (100.0 / pct_win)
        br_recent_pct = float(np.mean(ach_win < T) * 100.0)

        br_hist[t] = br_recent_pct

        # exponential multiplicative update (correct sign for your mapping)
        s = s * np.exp(kappa * (budget - br_recent_pct))
        s = float(np.clip(s, s_min, s_max))
        s_hist[t] = s

    # if your evaluator expects phase arrays, keep your existing conversion
    _, _, _, pct_eff, _ = avg_to_phase_arrays(pct_del, floors, split, PRIORITY)

    metrics = simulate_metrics_from_pct_eff(
        pct_eff=pct_eff,
        T=T,
        y_true=y_true,
        floors=PHASE_FLOORS,
        split=PHASE_SPLIT,
        priority=PRIORITY
    )
    debug = {"pct_cmd": pct_cmd, "pct_delivered": pct_del, "br_recent_pct": br_hist}
    return s_hist, pct_eff, metrics, debug



# Repeated-run experiment and confidence intervals

In [ ]:
# ==========================
# Cell 7 — Defensible Sweep + Robustness Runner (FIXED, SIMPLE, FULL)
# ==========================

import numpy as np
import pandas as pd

def run_defensible_sweep(art, exp_tag="run", include_oracles=True, include_block_oracle=True):
    """
    Simple, defensible sweep:
    - Uses TRAIN-only safety selection for ML fixed (via OOF preds).
    - Evaluates on TEST only.
    - Includes ML adaptive (deployable; uses forecasts + past outcomes).
    - Includes Oracle diagnostics (optional) clearly labeled.
    - Includes baselines (floors-only, part-night).
    """

    # --- unpack artifacts ---
    y_train = art["y_train"]     # train labels
    oof     = art["oof"]         # train out-of-fold preds
    resid   = art["resid"]       # |y_train - oof|
    y_true  = art["y_true"]      # test labels
    y_hat   = art["y_hat"]       # test preds (point forecasts)

    rows = []

    comfort_avg = time_weighted_floor(PHASE_FLOORS, PHASE_SPLIT)

    # --- baselines (independent of alpha/buffer/budget) ---
    for H in NIGHTS_H:
        T = float(H) * 3600.0
        n = len(y_true)

        # Floors-only baseline (comfort floors)
        pct_floor = np.full(n, comfort_avg, dtype=float)
        m_floor = simulate_literal_phase_schedule(
            schedule=PHASE_FLOORS,
            T=T,
            y_true=y_true,
            split=PHASE_SPLIT
        )
        rows.append({
            "exp_tag": exp_tag,
            "Night_h": float(H),
            "Budget": np.nan,
            "Scenario": "strict",
            "Policy": "Floors-only baseline",
            "alpha": np.nan,
            "buffer_mode": "baseline",
            "blackout_rate": m_floor["blackout_rate"],
            "energy_saving": m_floor["energy_saving"],
            "unmet_minutes_blackout_p90": m_floor["unmet_minutes_blackout_p90"],
            "Safety": np.nan,
        })

        # Part-night baseline (100/50/70) — uses average brightness proxy for energy
        # If you prefer phase-resolved evaluation, swap in avg_to_phase_arrays for this baseline too.
        _, _, _, bavg = schedule_baseline_avg_brightness(PART_NIGHT_SCHEDULE, PHASE_FLOORS, PHASE_SPLIT)
        pct_sched = np.full(n, float(bavg), dtype=float)

        m_sched = simulate_literal_phase_schedule(
            schedule=PART_NIGHT_SCHEDULE,
            T=T,
            y_true=y_true,
            split=PHASE_SPLIT
        )
        rows.append({
            "exp_tag": exp_tag,
            "Night_h": float(H),
            "Budget": np.nan,
            "Scenario": "strict",
            "Policy": "Part-night baseline (100/50/70)",
            "alpha": np.nan,
            "buffer_mode": "baseline",
            "blackout_rate": m_sched["blackout_rate"],
            "energy_saving": m_sched["energy_saving"],
            "unmet_minutes_blackout_p90": m_sched["unmet_minutes_blackout_p90"],
            "Safety": np.nan,
        })

    # --- main sweeps ---
    for alpha in ROBUST_ALPHAS:
        alpha = float(alpha)
        q_alpha = float(np.quantile(resid, 1.0 - alpha))  # train-only residual quantile

        for buffer_mode in ROBUST_BUFFER_MODES:
            buffer_mode = str(buffer_mode).lower()
            qbuf = 0.0 if buffer_mode == "none" else q_alpha  # <-- DO NOT overwrite later

            for H in NIGHTS_H:
                T = float(H) * 3600.0

                for bud in BUDGETS_EXT:
                    bud = float(bud)

                    # ==========================
                    # ML fixed (train-selected safety)
                    # ==========================
                    s_star, _, _ = choose_fixed_safety(
                        y_train, oof, T, qbuf, PHASE_FLOORS, PHASE_SPLIT,
                        budget=bud, s_grid=S_GRID
                    )
                    pct = ml_pct_from_forecast(y_hat, T, s_star, qbuf, PHASE_FLOORS, PHASE_SPLIT)
                    pct = apply_hysteresis_and_ramp(pct, HYSTERESIS, RAMP_LIMIT)
                    m = simulate_metrics_from_pct_eff(
                        pct_eff=pct,
                        T=T,
                        y_true=y_true,
                        floors=PHASE_FLOORS,
                        split=PHASE_SPLIT,
                        priority=PRIORITY
                    )

                    rows.append({
                        "exp_tag": exp_tag,
                        "Night_h": float(H),
                        "Budget": bud,
                        "Scenario": "strict",
                        "Policy": "ML fixed (train-selected)",
                        "alpha": alpha,
                        "buffer_mode": buffer_mode,
                        "blackout_rate": m["blackout_rate"],
                        "energy_saving": m["energy_saving"],
                        "unmet_minutes_blackout_p90": m["unmet_minutes_blackout_p90"],
                        "Safety": float(s_star),
                    })

                    # ==========================
                    # ML adaptive (deployable)
                    # ==========================
                    # Uses forecast + buffer to set brightness; uses realized outcomes (simulated via y_true)
                    # only as feedback for subsequent updates (no oracle look-ahead in the decision for t).
                    s_hist, pct_eff_adapt, m_adapt, dbg = adaptive_controller_exp(
                        y_true=y_true,
                        y_hat=y_hat,
                        T=T,
                        qbuf=qbuf,                 # respects buffer_mode
                        floors=PHASE_FLOORS,
                        split=PHASE_SPLIT,
                        budget=bud,                # <-- correct budget variable
                        window=ADAPT_WINDOW,
                        kappa=0.02,
                        s_init=SAFETY_INIT,
                        hysteresis=HYSTERESIS,
                        ramp_limit=RAMP_LIMIT,
                        s_min=S_MIN,
                        s_max=S_MAX
                    )

                    rows.append({
                        "exp_tag": exp_tag,
                        "Night_h": float(H),
                        "Budget": bud,
                        "Scenario": "strict",
                        "Policy": "ML adaptive",
                        "alpha": alpha,
                        "buffer_mode": buffer_mode,
                        "blackout_rate": m_adapt["blackout_rate"],
                        "energy_saving": m_adapt["energy_saving"],
                        "unmet_minutes_blackout_p90": m_adapt["unmet_minutes_blackout_p90"],
                        "Safety": np.nan,
                    })

                    # ==========================
                    # MPC-style baseline (deployable, optimization-based)
                    # ==========================
                    rho_star, _, _ = choose_mpc_reserve(
                        y_true=y_train,          # training-side labels
                        y_hat=oof,               # training-side OOF preds
                        T=T,
                        qbuf=qbuf,
                        floors=PHASE_FLOORS,
                        split=PHASE_SPLIT,
                        weights=PHASE_WEIGHTS,
                        budget=bud
                    )

                    pct_eff_mpc, m_mpc, dbg_mpc = mpc_phase_controller(
                        y_true=y_true,
                        y_hat=y_hat,
                        T=T,
                        qbuf=qbuf,
                        floors=PHASE_FLOORS,
                        split=PHASE_SPLIT,
                        weights=PHASE_WEIGHTS,
                        rho=rho_star,
                        smooth_output=False
                    )

                    rows.append({
                        "exp_tag": exp_tag,
                        "Night_h": float(H),
                        "Budget": bud,
                        "Scenario": "strict",
                        "Policy": MPC_POLICY_NAME,
                        "alpha": alpha,
                        "buffer_mode": buffer_mode,
                        "blackout_rate": m_mpc["blackout_rate"],
                        "energy_saving": m_mpc["energy_saving"],
                        "unmet_minutes_blackout_p90": m_mpc["unmet_minutes_blackout_p90"],
                        "Safety": float(rho_star),   # store rho here
                    })

                    # ==========================
                    # Oracles (diagnostic only)
                    # ==========================
                    if include_oracles:
                        s_or, _, m_or = choose_fixed_safety(
                            y_true, y_hat, T, qbuf, PHASE_FLOORS, PHASE_SPLIT,
                            budget=bud, s_grid=S_GRID
                        )
                        rows.append({
                            "exp_tag": exp_tag,
                            "Night_h": float(H),
                            "Budget": bud,
                            "Scenario": "strict",
                            "Policy": "Oracle fixed (diagnostic)",
                            "alpha": alpha,
                            "buffer_mode": buffer_mode,
                            "blackout_rate": m_or["blackout_rate"],
                            "energy_saving": m_or["energy_saving"],
                            "unmet_minutes_blackout_p90": m_or["unmet_minutes_blackout_p90"],
                            "Safety": float(s_or),
                        })

                        if include_block_oracle:
                            pct_blk, m_blk = block_fixed_controller(
                                y_true, y_hat, T, qbuf, PHASE_FLOORS, PHASE_SPLIT,
                                budget=bud, B=30
                            )
                            rows.append({
                                "exp_tag": exp_tag,
                                "Night_h": float(H),
                                "Budget": bud,
                                "Scenario": "strict",
                                "Policy": "Oracle block-fixed (diagnostic)",
                                "alpha": alpha,
                                "buffer_mode": buffer_mode,
                                "blackout_rate": m_blk["blackout_rate"],
                                "energy_saving": m_blk["energy_saving"],
                                "unmet_minutes_blackout_p90": m_blk["unmet_minutes_blackout_p90"],
                                "Safety": np.nan,
                            })

    return pd.DataFrame(rows)


# ==========================
# Robustness Runner (MODE-driven)
# ==========================

all_trades = []

models_to_run = ["rf"]
if CFG["run_catboost"] and CATBOOST_AVAILABLE:
    models_to_run.append("cat")

print("Models to run:", models_to_run)

for protocol in PROTOCOLS:
    for split_seed in CFG["split_seeds"]:
        df_train, df_test = make_split(
            df0,
            split_seed=split_seed,
            use_iqr=USE_IQR_CLEANING,
            protocol=protocol,
            early_frac=SHIFT_EARLY_FRAC,
            late_frac=SHIFT_LATE_FRAC,
            min_cycles=SHIFT_MIN_CYCLES,
            order_col=SHIFT_ORDER_COL
        )

        for model_seed in CFG["model_seeds"]:
            for model_name in models_to_run:
                exp_tag = (
                    f"{MODE}_{protocol}_split{split_seed}_{model_name}_seed{model_seed}"
                    f"_iqr{int(USE_IQR_CLEANING)}"
                )

                art = fit_predict(df_train, df_test, model_name=model_name, model_seed=model_seed)

                trade = run_defensible_sweep(
                    art,
                    exp_tag=exp_tag,
                    include_oracles=True,
                    include_block_oracle=True
                )

                trade["mode"] = MODE
                trade["protocol"] = protocol
                trade["split_seed"] = split_seed
                trade["model_seed"] = model_seed
                trade["model"] = model_name
                trade["use_iqr"] = bool(USE_IQR_CLEANING)
                trade["shift_early_frac"] = SHIFT_EARLY_FRAC if protocol == "aging_shift" else np.nan
                trade["shift_late_frac"]  = SHIFT_LATE_FRAC  if protocol == "aging_shift" else np.nan

                all_trades.append(trade)

trade_master = pd.concat(all_trades, ignore_index=True)
trade_master.to_csv("trade_master.csv", index=False)

print("\nSaved: trade_master.csv | rows:", len(trade_master))
display(trade_master.head(12))

In [ ]:
# ==========================================================
# R1 Major Comment 2:
# Statistical reliability of reported results
# Repeated grouped split / model-seed 95% confidence intervals
#
# Use this after trade_master has been created.
# Recommended final run:
#   MODE = "review_fast"
#   PROTOCOLS = ["standard"]
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from scipy.stats import t
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

OUTDIR = REVIEW_OUTDIR / "R1_major2_CI_outputs"
OUTDIR.mkdir(exist_ok=True)

# -----------------------------
# Main paper configuration
# -----------------------------
MAIN_PROTOCOL = "standard"
MAIN_MODEL = "rf"
MAIN_ALPHA = 0.20
MAIN_BUFFER = "fixed"

PAPER_POLICIES = [
    "Floors-only baseline",
    "Part-night baseline (100/50/70)",
    "ML fixed (train-selected)",
    "ML adaptive",
    "MPC-style baseline",
    "Oracle fixed (diagnostic)",
    "Oracle block-fixed (diagnostic)",
]

DEPLOYABLE_AND_ORACLE = [
    "ML fixed (train-selected)",
    "ML adaptive",
    "MPC-style baseline",
    "Oracle fixed (diagnostic)",
    "Oracle block-fixed (diagnostic)",
]

METRICS = [
    "blackout_rate",
    "energy_saving",
    "unmet_minutes_blackout_p90",
]

assert "trade_master" in globals(), "Run the main experiment first so trade_master exists."

df = trade_master.copy()

# ----------------------------------------------------------
# Filter to main manuscript protocol/model only
# ----------------------------------------------------------
df = df[
    (df["protocol"] == MAIN_PROTOCOL) &
    (df["model"] == MAIN_MODEL)
].copy()

# Keep baselines plus main residual-buffer setting.
is_baseline = df["buffer_mode"].eq("baseline")
is_main_setting = (
    df["buffer_mode"].eq(MAIN_BUFFER) &
    np.isclose(df["alpha"], MAIN_ALPHA)
)

df = df[is_baseline | is_main_setting].copy()
df = df[df["Policy"].isin(PAPER_POLICIES)].copy()

if df.empty:
    raise ValueError(
        "No rows after filtering. Check MAIN_PROTOCOL, MAIN_MODEL, MAIN_ALPHA, "
        "MAIN_BUFFER, and policy names."
    )

# ----------------------------------------------------------
# Avoid fake precision for baselines
# Baselines do not depend on model_seed, so repeated model_seed rows
# would duplicate the same baseline result.
# ----------------------------------------------------------
baseline_df = df[df["buffer_mode"].eq("baseline")].copy()
nonbaseline_df = df[~df["buffer_mode"].eq("baseline")].copy()

baseline_df = baseline_df.drop_duplicates(
    subset=[
        "protocol",
        "split_seed",
        "Scenario",
        "Night_h",
        "Budget",
        "Policy",
        "buffer_mode",
    ]
)

# Replicate ID:
# - baselines: split_seed only
# - model-driven policies: split_seed × model_seed
baseline_df["replicate_id"] = baseline_df["split_seed"].astype(str)
nonbaseline_df["replicate_id"] = (
    nonbaseline_df["split_seed"].astype(str) + "_" +
    nonbaseline_df["model_seed"].astype(str)
)

df_ci = pd.concat([baseline_df, nonbaseline_df], ignore_index=True)

# ----------------------------------------------------------
# Diagnostics on repeated-run design
# ----------------------------------------------------------
split_seeds_used = sorted([int(x) for x in df_ci["split_seed"].dropna().unique()])
model_seeds_used = sorted([int(x) for x in df_ci["model_seed"].dropna().unique()])
n_split = len(split_seeds_used)
n_model = len(model_seeds_used)

print("Rows used for CI:", len(df_ci))
print("Split seeds:", split_seeds_used)
print("Model seeds:", model_seeds_used)
print("Policies:", sorted(df_ci["Policy"].unique()))
print("Scenarios:", sorted(df_ci["Scenario"].dropna().unique()))
print(f"\nRepeated-run design: {n_split} split seeds × {n_model} model seeds")

if n_split < 5:
    print(
        "WARNING: Fewer than 5 split seeds detected. "
        "This is okay for debugging but weak for the manuscript."
    )

if n_model < 2:
    print(
        "WARNING: Fewer than 2 model seeds detected. "
        "This does not fully address stochastic model-training variability."
    )

# ==========================================================
# Helper functions
# ==========================================================

def mean_ci95(x):
    """
    Mean and t-based 95% CI across repeated runs.
    """
    x = pd.Series(x).dropna().astype(float)
    n = len(x)

    if n == 0:
        return pd.Series({
            "mean": np.nan,
            "std": np.nan,
            "n": 0,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "ci95_half_width": np.nan,
        })

    mean = float(x.mean())
    std = float(x.std(ddof=1)) if n > 1 else 0.0

    if n > 1:
        crit = float(t.ppf(0.975, df=n - 1)) if SCIPY_AVAILABLE else 1.96
        half_width = crit * std / np.sqrt(n)
    else:
        half_width = 0.0

    return pd.Series({
        "mean": mean,
        "std": std,
        "n": n,
        "ci95_low": mean - half_width,
        "ci95_high": mean + half_width,
        "ci95_half_width": half_width,
    })


def summarize_ci(data, group_cols, metrics):
    rows = []

    for keys, g in data.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = dict(zip(group_cols, keys))

        for metric in metrics:
            stats = mean_ci95(g[metric])
            row = base.copy()
            row["metric"] = metric
            row.update(stats.to_dict())
            rows.append(row)

    return pd.DataFrame(rows)


def wide_ci_table(ci_long, index_cols):
    wide = ci_long.pivot_table(
        index=index_cols,
        columns="metric",
        values=["mean", "std", "ci95_low", "ci95_high", "ci95_half_width", "n"],
        aggfunc="first",
    ).reset_index()

    wide.columns = [
        "_".join([str(c) for c in col if str(c) != ""]).strip("_")
        for col in wide.columns
    ]

    return wide


def format_mean_ci(mean, low, high, digits=2):
    if pd.isna(mean):
        return ""
    return f"{mean:.{digits}f} [{low:.{digits}f}, {high:.{digits}f}]"


def add_formatted_ci_columns(wide_df, metric_names, digits=2):
    out = wide_df.copy()

    for metric in metric_names:
        mean_col = f"mean_{metric}"
        low_col = f"ci95_low_{metric}"
        high_col = f"ci95_high_{metric}"

        if all(c in out.columns for c in [mean_col, low_col, high_col]):
            out[f"{metric}_mean_95CI"] = out.apply(
                lambda r: format_mean_ci(
                    r[mean_col],
                    r[low_col],
                    r[high_col],
                    digits=digits,
                ),
                axis=1,
            )

    return out


# ==========================================================
# A) Table 3 replacement:
# BR_min(H) under floors-only baseline with 95% CI
# ==========================================================

brmin_df = df_ci[
    (df_ci["Scenario"].eq("strict")) &
    (df_ci["Policy"].eq("Floors-only baseline"))
].copy()

if brmin_df.empty:
    raise ValueError("No floors-only baseline rows found for Table 3 CI.")

table3_ci_long = summarize_ci(
    brmin_df,
    group_cols=["Night_h"],
    metrics=["blackout_rate", "energy_saving", "unmet_minutes_blackout_p90"],
)

table3_ci = wide_ci_table(table3_ci_long, ["Night_h"])
table3_ci = table3_ci.sort_values("Night_h").reset_index(drop=True)

table3_ci["BR_min_95CI_text"] = table3_ci.apply(
    lambda r: format_mean_ci(
        r["mean_blackout_rate"],
        r["ci95_low_blackout_rate"],
        r["ci95_high_blackout_rate"],
        digits=2,
    ),
    axis=1,
)

table3_ci["UMBO_p90_95CI_text"] = table3_ci.apply(
    lambda r: format_mean_ci(
        r["mean_unmet_minutes_blackout_p90"],
        r["ci95_low_unmet_minutes_blackout_p90"],
        r["ci95_high_unmet_minutes_blackout_p90"],
        digits=2,
    ),
    axis=1,
)

table3_ci.to_csv(
    OUTDIR / "Table3_BRmin_floors_only_95CI.csv",
    index=False,
)

table3_manuscript = table3_ci[
    [
        "Night_h",
        "BR_min_95CI_text",
        "UMBO_p90_95CI_text",
        "n_blackout_rate",
    ]
].copy()

table3_manuscript = table3_manuscript.rename(columns={
    "Night_h": "H",
    "BR_min_95CI_text": "BR_min_mean_95CI",
    "UMBO_p90_95CI_text": "UMBO_p90_mean_95CI",
    "n_blackout_rate": "n_repeats",
})

table3_manuscript.to_csv(
    OUTDIR / "Table3_manuscript_ready_BRmin_95CI.csv",
    index=False,
)

print("\n=== Table 3 replacement: BR_min(H) with 95% CI ===")
display(table3_manuscript)


# ==========================================================
# B) Main Section X result curves:
# BR, ES, and UM|BO p90 with 95% CI
# ==========================================================

curve_df = df_ci[df_ci["Budget"].notna()].copy()

if curve_df.empty:
    raise ValueError("No budgeted rows found for Section X curve CI.")

curve_ci_long = summarize_ci(
    curve_df,
    group_cols=["Scenario", "Night_h", "Budget", "Policy"],
    metrics=METRICS,
)

curve_ci = wide_ci_table(
    curve_ci_long,
    ["Scenario", "Night_h", "Budget", "Policy"],
)

curve_ci = curve_ci.sort_values(
    ["Scenario", "Night_h", "Budget", "Policy"]
).reset_index(drop=True)

curve_ci.to_csv(
    OUTDIR / "SectionX_curves_BR_ES_UMBOp90_95CI.csv",
    index=False,
)

curve_ci_formatted = add_formatted_ci_columns(
    curve_ci,
    metric_names=[
        "blackout_rate",
        "energy_saving",
        "unmet_minutes_blackout_p90",
    ],
    digits=2,
)

curve_ci_formatted.to_csv(
    OUTDIR / "SectionX_curves_manuscript_ready_mean_95CI.csv",
    index=False,
)

print("\n=== Section X curve values with 95% CI ===")
display(curve_ci_formatted.head(40))


# ==========================================================
# C) Alignment check:
# BR - BR_min(H), supporting statements such as
# "aligning with BR_min(1.0 h)"
# ==========================================================

floor_by_split = brmin_df[
    ["split_seed", "Night_h", "blackout_rate"]
].drop_duplicates(
    subset=["split_seed", "Night_h"]
).rename(
    columns={"blackout_rate": "BR_min_floors"}
)

policy_df = curve_df[
    curve_df["Policy"].isin(DEPLOYABLE_AND_ORACLE)
].copy()

policy_vs_floor = policy_df.merge(
    floor_by_split,
    on=["split_seed", "Night_h"],
    how="left",
)

if policy_vs_floor["BR_min_floors"].isna().any():
    missing_count = int(policy_vs_floor["BR_min_floors"].isna().sum())
    print(f"WARNING: {missing_count} rows missing BR_min_floors after merge.")

policy_vs_floor["BR_minus_BRmin"] = (
    policy_vs_floor["blackout_rate"] - policy_vs_floor["BR_min_floors"]
)

alignment_ci_long = summarize_ci(
    policy_vs_floor,
    group_cols=["Scenario", "Night_h", "Budget", "Policy"],
    metrics=["BR_minus_BRmin"],
)

alignment_ci = wide_ci_table(
    alignment_ci_long,
    ["Scenario", "Night_h", "Budget", "Policy"],
)

alignment_ci = alignment_ci.sort_values(
    ["Scenario", "Night_h", "Budget", "Policy"]
).reset_index(drop=True)

alignment_ci["BR_minus_BRmin_95CI_text"] = alignment_ci.apply(
    lambda r: format_mean_ci(
        r["mean_BR_minus_BRmin"],
        r["ci95_low_BR_minus_BRmin"],
        r["ci95_high_BR_minus_BRmin"],
        digits=2,
    ),
    axis=1,
)

alignment_ci.to_csv(
    OUTDIR / "Alignment_policy_BR_minus_BRmin_95CI.csv",
    index=False,
)

print("\n=== Alignment with BR_min(H): BR - BR_min with 95% CI ===")
display(alignment_ci.head(40))


# ==========================================================
# D) Plots with 95% CI error bars
# These can replace deterministic Figures 3-5 or be used as appendix figures.
# ==========================================================

def plot_br_vs_budget_ci(ci_wide, H, scenario="strict", policies=None):
    s = ci_wide[
        (np.isclose(ci_wide["Night_h"], H)) &
        (ci_wide["Scenario"].eq(scenario))
    ].copy()

    if policies is not None:
        s = s[s["Policy"].isin(policies)].copy()

    if s.empty:
        print(f"No rows for H={H}, scenario={scenario}")
        return

    plt.figure(figsize=(7, 5))

    for pol in sorted(s["Policy"].unique()):
        tpol = s[s["Policy"].eq(pol)].sort_values("Budget")

        plt.errorbar(
            tpol["Budget"],
            tpol["mean_blackout_rate"],
            yerr=tpol["ci95_half_width_blackout_rate"],
            marker="o",
            capsize=4,
            label=pol,
        )

    xs = sorted(s["Budget"].dropna().unique())
    plt.plot(xs, xs, linestyle="--", label="Budget line y=x")

    plt.title(f"Blackout rate vs budget with 95% CI, H={H}h, {scenario}")
    plt.xlabel("Blackout budget B (%)")
    plt.ylabel("Blackout rate BR (%)")
    plt.grid(True)
    plt.legend(fontsize=8)
    plt.tight_layout()

    fname = OUTDIR / f"Fig_BR_vs_budget_CI_H{str(H).replace('.', '_')}_{scenario}.png"
    plt.savefig(fname, dpi=300)
    plt.show()


def plot_es_vs_br_ci(ci_wide, H, scenario="strict", policies=None):
    s = ci_wide[
        (np.isclose(ci_wide["Night_h"], H)) &
        (ci_wide["Scenario"].eq(scenario))
    ].copy()

    if policies is not None:
        s = s[s["Policy"].isin(policies)].copy()

    if s.empty:
        print(f"No rows for H={H}, scenario={scenario}")
        return

    plt.figure(figsize=(7, 5))

    for pol in sorted(s["Policy"].unique()):
        tpol = s[s["Policy"].eq(pol)].sort_values("Budget")

        plt.errorbar(
            tpol["mean_blackout_rate"],
            tpol["mean_energy_saving"],
            xerr=tpol["ci95_half_width_blackout_rate"],
            yerr=tpol["ci95_half_width_energy_saving"],
            marker="o",
            capsize=4,
            label=pol,
        )

    plt.title(f"Energy saving vs blackout rate with 95% CI, H={H}h, {scenario}")
    plt.xlabel("Blackout rate BR (%)")
    plt.ylabel("Energy saving ES (%)")
    plt.grid(True)
    plt.legend(fontsize=8)
    plt.tight_layout()

    fname = OUTDIR / f"Fig_ES_vs_BR_CI_H{str(H).replace('.', '_')}_{scenario}.png"
    plt.savefig(fname, dpi=300)
    plt.show()


def plot_severity_vs_br_ci(ci_wide, H, scenario="strict", policies=None):
    s = ci_wide[
        (np.isclose(ci_wide["Night_h"], H)) &
        (ci_wide["Scenario"].eq(scenario))
    ].copy()

    if policies is not None:
        s = s[s["Policy"].isin(policies)].copy()

    if s.empty:
        print(f"No rows for H={H}, scenario={scenario}")
        return

    plt.figure(figsize=(7, 5))

    for pol in sorted(s["Policy"].unique()):
        tpol = s[s["Policy"].eq(pol)].sort_values("Budget")

        plt.errorbar(
            tpol["mean_blackout_rate"],
            tpol["mean_unmet_minutes_blackout_p90"],
            xerr=tpol["ci95_half_width_blackout_rate"],
            yerr=tpol["ci95_half_width_unmet_minutes_blackout_p90"],
            marker="o",
            capsize=4,
            label=pol,
        )

    plt.title(f"UM|BO p90 vs blackout rate with 95% CI, H={H}h, {scenario}")
    plt.xlabel("Blackout rate BR (%)")
    plt.ylabel("UM|BO p90 (minutes)")
    plt.grid(True)
    plt.legend(fontsize=8)
    plt.tight_layout()

    fname = OUTDIR / f"Fig_UMBOp90_vs_BR_CI_H{str(H).replace('.', '_')}_{scenario}.png"
    plt.savefig(fname, dpi=300)
    plt.show()


for scenario in sorted(curve_ci["Scenario"].dropna().unique()):
    for H in sorted(curve_ci["Night_h"].dropna().unique()):
        plot_br_vs_budget_ci(
            curve_ci,
            H=H,
            scenario=scenario,
            policies=DEPLOYABLE_AND_ORACLE,
        )

        plot_es_vs_br_ci(
            curve_ci,
            H=H,
            scenario=scenario,
            policies=DEPLOYABLE_AND_ORACLE,
        )

        plot_severity_vs_br_ci(
            curve_ci,
            H=H,
            scenario=scenario,
            policies=DEPLOYABLE_AND_ORACLE,
        )


# ==========================================================
# E) Save repeated-run design note for reviewer response
# ==========================================================

design_note = {
    "main_protocol": MAIN_PROTOCOL,
    "main_model": MAIN_MODEL,
    "main_alpha": MAIN_ALPHA,
    "main_buffer": MAIN_BUFFER,
    "n_split_seeds": int(n_split),
    "split_seeds": split_seeds_used,
    "n_model_seeds": int(n_model),
    "model_seeds": model_seeds_used,
    "n_rows_used_for_ci": int(len(df_ci)),
    "ci_method": "mean ± t-based 95% confidence interval across repeated grouped split/model-seed runs",
    "baseline_note": "Baseline rows were deduplicated by split seed because baselines do not depend on model seed.",
}

pd.Series(design_note).to_csv(
    OUTDIR / "R1_major2_repeated_run_design_note.csv"
)

print("\nRepeated-run design note:")
for k, v in design_note.items():
    print(f"{k}: {v}")

print("\nSaved outputs to:", OUTDIR.resolve())

# Reviewer validation analyses

The following cells reproduce the targeted validation analyses added during revision.


In [ ]:
# ==========================================================
# LEAN REVIEW RUNNER
# For:
# R1 Major Comment 3: no-buffer vs fixed-buffer ablation
# R2 Major Comment 10: UM|BO p75 / p90 / p95 sensitivity
#
# Much faster than the previous Cell A.
# ==========================================================

import numpy as np
import pandas as pd
from pathlib import Path
from time import time

# REVIEW_OUTDIR is defined in the configuration cell above.
REVIEW_OUTDIR.mkdir(exist_ok=True)

REVIEW_MODE = "review_lean"

# Use [0, 1, 2] first. If you have time later, expand to [0,1,2,3,4].
REVIEW_SPLIT_SEEDS = [0, 1, 2]
REVIEW_MODEL_SEEDS = [0]

REVIEW_PROTOCOLS = ["standard"]
REVIEW_MODELS = ["rf"]

REVIEW_ALPHAS = [0.20]
REVIEW_BUFFER_MODES = ["none", "fixed"]

# Use manuscript budgets only to save time.
REVIEW_BUDGETS = [2, 5, 10]
REVIEW_HORIZONS = NIGHTS_H

REVIEW_POLICIES = [
    "ML fixed (train-selected)",
    "ML adaptive",
]

def _severity_from_metric_dict(m):
    blackout = np.asarray(m.get("blackout_mask", []), dtype=bool)
    unmet = np.asarray(m.get("unmet_minutes_per_night", []), dtype=float)

    if len(unmet) > 0 and len(blackout) == len(unmet):
        vals = unmet[blackout & (unmet > 0)]
    else:
        vals = np.array([], dtype=float)

    if len(vals) == 0:
        return {
            "unmet_minutes_blackout_p50": 0.0,
            "unmet_minutes_blackout_p75": 0.0,
            "unmet_minutes_blackout_p90": float(m.get("unmet_minutes_blackout_p90", 0.0)),
            "unmet_minutes_blackout_p95": 0.0,
        }

    return {
        "unmet_minutes_blackout_p50": float(np.percentile(vals, 50)),
        "unmet_minutes_blackout_p75": float(np.percentile(vals, 75)),
        "unmet_minutes_blackout_p90": float(np.percentile(vals, 90)),
        "unmet_minutes_blackout_p95": float(np.percentile(vals, 95)),
    }


def _metric_columns(m):
    sev = _severity_from_metric_dict(m)

    return {
        "blackout_rate": float(m["blackout_rate"]),
        "energy_saving": float(m["energy_saving"]),
        "unmet_minutes_blackout": float(m.get("unmet_minutes_blackout", np.nan)),
        "unmet_minutes_blackout_p50": sev["unmet_minutes_blackout_p50"],
        "unmet_minutes_blackout_p75": sev["unmet_minutes_blackout_p75"],
        "unmet_minutes_blackout_p90": sev["unmet_minutes_blackout_p90"],
        "unmet_minutes_blackout_p95": sev["unmet_minutes_blackout_p95"],
        "unmet_minutes_all_mean": float(m.get("unmet_minutes_all_mean", np.nan)),
    }


def _make_result_row(
    exp_tag,
    H,
    budget,
    scenario,
    policy,
    alpha,
    buffer_mode,
    metrics,
    safety=np.nan,
):
    row = {
        "exp_tag": exp_tag,
        "Night_h": float(H),
        "Budget": np.nan if budget is None else float(budget),
        "Scenario": scenario,
        "Policy": policy,
        "alpha": np.nan if alpha is None else float(alpha),
        "buffer_mode": buffer_mode,
        "Safety": safety,
    }

    row.update(_metric_columns(metrics))
    return row


def run_defensible_sweep_review_lean(art, exp_tag="review_lean_run"):
    y_train = art["y_train"]
    oof = art["oof"]
    resid = art["resid"]
    y_true = art["y_true"]
    y_hat = art["y_hat"]

    rows = []

    for alpha in REVIEW_ALPHAS:
        alpha = float(alpha)
        q_alpha = float(np.quantile(resid, 1.0 - alpha))

        for buffer_mode in REVIEW_BUFFER_MODES:
            buffer_mode = str(buffer_mode).lower()
            qbuf = 0.0 if buffer_mode == "none" else q_alpha

            for H in REVIEW_HORIZONS:
                T = float(H) * 3600.0

                for bud in REVIEW_BUDGETS:
                    bud = float(bud)

                    # --------------------------
                    # ML fixed
                    # --------------------------
                    s_star, _, _ = choose_fixed_safety(
                        y_train,
                        oof,
                        T,
                        qbuf,
                        PHASE_FLOORS,
                        PHASE_SPLIT,
                        budget=bud,
                        s_grid=S_GRID,
                    )

                    pct = ml_pct_from_forecast(
                        y_hat,
                        T,
                        s_star,
                        qbuf,
                        PHASE_FLOORS,
                        PHASE_SPLIT,
                    )

                    pct = apply_hysteresis_and_ramp(
                        pct,
                        HYSTERESIS,
                        RAMP_LIMIT,
                    )

                    m_fixed = simulate_metrics_from_pct_eff(
                        pct_eff=pct,
                        T=T,
                        y_true=y_true,
                        floors=PHASE_FLOORS,
                        split=PHASE_SPLIT,
                        priority=PRIORITY,
                    )

                    rows.append(
                        _make_result_row(
                            exp_tag=exp_tag,
                            H=H,
                            budget=bud,
                            scenario="strict",
                            policy="ML fixed (train-selected)",
                            alpha=alpha,
                            buffer_mode=buffer_mode,
                            metrics=m_fixed,
                            safety=float(s_star),
                        )
                    )

                    # --------------------------
                    # ML adaptive
                    # --------------------------
                    s_hist, pct_eff_adapt, m_adapt, dbg = adaptive_controller_exp(
                        y_true=y_true,
                        y_hat=y_hat,
                        T=T,
                        qbuf=qbuf,
                        floors=PHASE_FLOORS,
                        split=PHASE_SPLIT,
                        budget=bud,
                        window=ADAPT_WINDOW,
                        kappa=0.02,
                        s_init=SAFETY_INIT,
                        hysteresis=HYSTERESIS,
                        ramp_limit=RAMP_LIMIT,
                        s_min=S_MIN,
                        s_max=S_MAX,
                    )

                    rows.append(
                        _make_result_row(
                            exp_tag=exp_tag,
                            H=H,
                            budget=bud,
                            scenario="strict",
                            policy="ML adaptive",
                            alpha=alpha,
                            buffer_mode=buffer_mode,
                            metrics=m_adapt,
                            safety=np.nan,
                        )
                    )

    return pd.DataFrame(rows)


# ==========================================================
# Execute lean review run
# ==========================================================

start = time()
all_review_trades = []

print("Starting LEAN review run...")
print("Protocols:", REVIEW_PROTOCOLS)
print("Split seeds:", REVIEW_SPLIT_SEEDS)
print("Model seeds:", REVIEW_MODEL_SEEDS)
print("Models:", REVIEW_MODELS)
print("Alphas:", REVIEW_ALPHAS)
print("Buffer modes:", REVIEW_BUFFER_MODES)
print("Budgets:", REVIEW_BUDGETS)
print("Policies:", REVIEW_POLICIES)

for protocol in REVIEW_PROTOCOLS:
    for split_seed in REVIEW_SPLIT_SEEDS:
        df_train_i, df_test_i = make_split(
            df0,
            split_seed=split_seed,
            use_iqr=USE_IQR_CLEANING,
            protocol=protocol,
            early_frac=SHIFT_EARLY_FRAC,
            late_frac=SHIFT_LATE_FRAC,
            min_cycles=SHIFT_MIN_CYCLES,
            order_col=SHIFT_ORDER_COL,
        )

        for model_seed in REVIEW_MODEL_SEEDS:
            for model_name in REVIEW_MODELS:
                exp_tag = (
                    f"{REVIEW_MODE}_{protocol}_split{split_seed}_"
                    f"{model_name}_seed{model_seed}_iqr{int(USE_IQR_CLEANING)}"
                )

                print("Running:", exp_tag)

                art_i = fit_predict(
                    df_train_i,
                    df_test_i,
                    model_name=model_name,
                    model_seed=model_seed,
                )

                trade_i = run_defensible_sweep_review_lean(
                    art_i,
                    exp_tag=exp_tag,
                )

                trade_i["mode"] = REVIEW_MODE
                trade_i["protocol"] = protocol
                trade_i["split_seed"] = split_seed
                trade_i["model_seed"] = model_seed
                trade_i["model"] = model_name
                trade_i["use_iqr"] = bool(USE_IQR_CLEANING)
                trade_i["shift_early_frac"] = SHIFT_EARLY_FRAC if protocol == "aging_shift" else np.nan
                trade_i["shift_late_frac"] = SHIFT_LATE_FRAC if protocol == "aging_shift" else np.nan

                all_review_trades.append(trade_i)

trade_master_review = pd.concat(all_review_trades, ignore_index=True)

out_csv = REVIEW_OUTDIR / "trade_master_review.csv"
trade_master_review.to_csv(out_csv, index=False)

elapsed_min = (time() - start) / 60.0

print("\nSaved:", out_csv)
print("Rows:", len(trade_master_review))
print("Elapsed minutes:", round(elapsed_min, 2))
print("Columns:", list(trade_master_review.columns))
display(trade_master_review.head(12))

In [ ]:
# ==========================================================
# R1 Major Comment 3B:
# Isolated residual-buffer effect
# Same model, same split, same safety factor.
# Only qbuf changes: qbuf = 0 vs qbuf = residual quantile.
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from scipy.stats import ttest_1samp, wilcoxon, t
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

OUTDIR = REVIEW_OUTDIR / "R1_major3_isolated_buffer_effect"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Debug settings:
# REVIEW_SPLIT_SEEDS_ISO = [0, 1, 2]
# REVIEW_MODEL_SEEDS_ISO = [0]

# Stronger final settings:
REVIEW_SPLIT_SEEDS_ISO = [0, 1, 2, 3, 4]
REVIEW_MODEL_SEEDS_ISO = [0, 1]

REVIEW_ALPHA = 0.20
REVIEW_BUDGETS_ISO = [2, 5, 10]
REVIEW_HORIZONS_ISO = NIGHTS_H

def _severity_from_metric_dict_iso(m):
    blackout = np.asarray(m.get("blackout_mask", []), dtype=bool)
    unmet = np.asarray(m.get("unmet_minutes_per_night", []), dtype=float)

    if len(unmet) > 0 and len(blackout) == len(unmet):
        vals = unmet[blackout & (unmet > 0)]
    else:
        vals = np.array([], dtype=float)

    if len(vals) == 0:
        return {
            "unmet_minutes_blackout_p75": 0.0,
            "unmet_minutes_blackout_p90": float(m.get("unmet_minutes_blackout_p90", 0.0)),
            "unmet_minutes_blackout_p95": 0.0,
        }

    return {
        "unmet_minutes_blackout_p75": float(np.percentile(vals, 75)),
        "unmet_minutes_blackout_p90": float(np.percentile(vals, 90)),
        "unmet_minutes_blackout_p95": float(np.percentile(vals, 95)),
    }


def _metric_cols_iso(m):
    sev = _severity_from_metric_dict_iso(m)
    return {
        "blackout_rate": float(m["blackout_rate"]),
        "energy_saving": float(m["energy_saving"]),
        "unmet_minutes_blackout": float(m.get("unmet_minutes_blackout", np.nan)),
        "unmet_minutes_blackout_p75": sev["unmet_minutes_blackout_p75"],
        "unmet_minutes_blackout_p90": sev["unmet_minutes_blackout_p90"],
        "unmet_minutes_blackout_p95": sev["unmet_minutes_blackout_p95"],
        "unmet_minutes_all_mean": float(m.get("unmet_minutes_all_mean", np.nan)),
    }


def ci95(x):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)

    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, 0

    mean = float(x.mean())
    std = float(x.std(ddof=1)) if n > 1 else 0.0

    if n > 1:
        crit = float(t.ppf(0.975, df=n - 1)) if SCIPY_AVAILABLE else 1.96
        half_width = crit * std / np.sqrt(n)
    else:
        half_width = 0.0

    return mean, mean - half_width, mean + half_width, half_width, n


def pvalues_delta(x):
    x = pd.Series(x).dropna().astype(float)

    if len(x) < 2 or not SCIPY_AVAILABLE:
        return np.nan, np.nan

    try:
        p_t = float(ttest_1samp(x, popmean=0.0).pvalue)
    except Exception:
        p_t = np.nan

    try:
        if np.allclose(x, 0):
            p_w = 1.0
        else:
            p_w = float(wilcoxon(x).pvalue)
    except Exception:
        p_w = np.nan

    return p_t, p_w


rows = []

for split_seed in REVIEW_SPLIT_SEEDS_ISO:
    df_train_i, df_test_i = make_split(
        df0,
        split_seed=split_seed,
        use_iqr=USE_IQR_CLEANING,
        protocol="standard",
        early_frac=SHIFT_EARLY_FRAC,
        late_frac=SHIFT_LATE_FRAC,
        min_cycles=SHIFT_MIN_CYCLES,
        order_col=SHIFT_ORDER_COL,
    )

    for model_seed in REVIEW_MODEL_SEEDS_ISO:
        exp_tag = f"isolated_buffer_split{split_seed}_rf_seed{model_seed}"
        print("Running:", exp_tag)

        art = fit_predict(
            df_train_i,
            df_test_i,
            model_name="rf",
            model_seed=model_seed,
        )

        y_train = art["y_train"]
        oof = art["oof"]
        resid = art["resid"]
        y_true = art["y_true"]
        y_hat = art["y_hat"]

        # Current manuscript buffer definition
        qbuf_fixed = float(np.quantile(resid, 1.0 - REVIEW_ALPHA))
        qbuf_none = 0.0

        for H in REVIEW_HORIZONS_ISO:
            T = float(H) * 3600.0

            for bud in REVIEW_BUDGETS_ISO:
                bud = float(bud)

                # ------------------------------------------------------
                # Select safety factor ONCE using no-buffer OOF.
                # Then evaluate same s_star with qbuf = 0 and qbuf = q.
                # This isolates the direct residual-buffer effect.
                # ------------------------------------------------------
                s_star, _, _ = choose_fixed_safety(
                    y_train,
                    oof,
                    T,
                    qbuf_none,
                    PHASE_FLOORS,
                    PHASE_SPLIT,
                    budget=bud,
                    s_grid=S_GRID,
                )

                for label, qbuf in [
                    ("no_buffer_same_s", qbuf_none),
                    ("buffer_same_s", qbuf_fixed),
                ]:
                    pct = ml_pct_from_forecast(
                        y_hat,
                        T,
                        s_star,
                        qbuf,
                        PHASE_FLOORS,
                        PHASE_SPLIT,
                    )

                    pct = apply_hysteresis_and_ramp(
                        pct,
                        HYSTERESIS,
                        RAMP_LIMIT,
                    )

                    m = simulate_metrics_from_pct_eff(
                        pct_eff=pct,
                        T=T,
                        y_true=y_true,
                        floors=PHASE_FLOORS,
                        split=PHASE_SPLIT,
                        priority=PRIORITY,
                    )

                    row = {
                        "exp_tag": exp_tag,
                        "split_seed": split_seed,
                        "model_seed": model_seed,
                        "replicate_id": f"{split_seed}_{model_seed}",
                        "Night_h": float(H),
                        "Budget": bud,
                        "Policy": "ML fixed (same safety factor)",
                        "alpha": REVIEW_ALPHA,
                        "condition": label,
                        "qbuf": float(qbuf),
                        "Safety": float(s_star),
                    }
                    row.update(_metric_cols_iso(m))
                    rows.append(row)

iso = pd.DataFrame(rows)

if iso.empty:
    raise ValueError("iso is empty. Check the upstream pipeline and evaluation settings.")

iso.to_csv(
    OUTDIR / "raw_isolated_buffer_same_safety_factor.csv",
    index=False,
)

print("Raw isolated rows:", len(iso))
print("Conditions:", iso["condition"].value_counts().to_dict())
print("Replicates:", iso["replicate_id"].nunique())

# Pair no buffer vs buffer
pair_cols = [
    "replicate_id",
    "split_seed",
    "model_seed",
    "Night_h",
    "Budget",
    "Policy",
]

no_df = iso[iso["condition"] == "no_buffer_same_s"].copy()
buf_df = iso[iso["condition"] == "buffer_same_s"].copy()

no_df = no_df[pair_cols + [
    "blackout_rate",
    "energy_saving",
    "unmet_minutes_blackout_p90",
]].rename(columns={
    "blackout_rate": "blackout_rate_no_buffer",
    "energy_saving": "energy_saving_no_buffer",
    "unmet_minutes_blackout_p90": "unmet_minutes_blackout_p90_no_buffer",
})

buf_df = buf_df[pair_cols + [
    "blackout_rate",
    "energy_saving",
    "unmet_minutes_blackout_p90",
]].rename(columns={
    "blackout_rate": "blackout_rate_buffer",
    "energy_saving": "energy_saving_buffer",
    "unmet_minutes_blackout_p90": "unmet_minutes_blackout_p90_buffer",
})

paired_iso = buf_df.merge(no_df, on=pair_cols, how="inner")

if paired_iso.empty:
    raise ValueError(
        "No paired rows found. Check that both no_buffer_same_s and buffer_same_s "
        "exist for the same split/model/horizon/budget/policy combinations."
    )

paired_iso["delta_blackout_rate"] = (
    paired_iso["blackout_rate_buffer"] - paired_iso["blackout_rate_no_buffer"]
)
paired_iso["delta_energy_saving"] = (
    paired_iso["energy_saving_buffer"] - paired_iso["energy_saving_no_buffer"]
)
paired_iso["delta_unmet_minutes_blackout_p90"] = (
    paired_iso["unmet_minutes_blackout_p90_buffer"] -
    paired_iso["unmet_minutes_blackout_p90_no_buffer"]
)

paired_iso.to_csv(
    OUTDIR / "paired_isolated_buffer_same_safety_factor.csv",
    index=False,
)

print("Paired rows:", len(paired_iso))

summary_rows = []

for keys, g in paired_iso.groupby(["Night_h", "Budget", "Policy"], dropna=False):
    H, B, policy = keys

    row = {
        "Night_h": H,
        "Budget": B,
        "Policy": policy,
        "n_pairs": len(g),
    }

    for metric in [
        "blackout_rate",
        "energy_saving",
        "unmet_minutes_blackout_p90",
    ]:
        no_col = f"{metric}_no_buffer"
        buf_col = f"{metric}_buffer"
        delta_col = f"delta_{metric}"

        no_mean, no_low, no_high, no_hw, _ = ci95(g[no_col])
        buf_mean, buf_low, buf_high, buf_hw, _ = ci95(g[buf_col])
        delta_mean, delta_low, delta_high, delta_hw, _ = ci95(g[delta_col])
        p_t, p_w = pvalues_delta(g[delta_col])

        row[f"{metric}_no_buffer_mean"] = no_mean
        row[f"{metric}_buffer_mean"] = buf_mean
        row[f"{metric}_delta_mean"] = delta_mean
        row[f"{metric}_delta_ci95_low"] = delta_low
        row[f"{metric}_delta_ci95_high"] = delta_high
        row[f"{metric}_delta_ci95_half_width"] = delta_hw
        row[f"{metric}_paired_t_p"] = p_t
        row[f"{metric}_wilcoxon_p"] = p_w

    summary_rows.append(row)

isolated_buffer_table = pd.DataFrame(summary_rows)
isolated_buffer_table = isolated_buffer_table.sort_values(["Night_h", "Budget"]).reset_index(drop=True)

isolated_buffer_table.to_csv(
    OUTDIR / "table_isolated_buffer_effect_same_safety_factor_95CI_pvalues.csv",
    index=False,
)

display(isolated_buffer_table)

# Plot delta BR. Negative means buffer reduces BR.
for H in sorted(isolated_buffer_table["Night_h"].unique()):
    dH = isolated_buffer_table[np.isclose(isolated_buffer_table["Night_h"], H)].copy()
    dH = dH.sort_values("Budget")

    plt.figure(figsize=(7, 5))

    plt.errorbar(
        dH["Budget"],
        dH["blackout_rate_delta_mean"],
        yerr=dH["blackout_rate_delta_ci95_half_width"],
        marker="o",
        capsize=4,
        label="ML fixed, same safety factor",
    )

    plt.axhline(0, linestyle="--")
    plt.xlabel("Blackout budget B (%)")
    plt.ylabel("Delta BR = BR(buffer) - BR(no buffer) (%)")
    plt.title(f"Isolated residual-buffer effect on BR, H={H}h")
    plt.legend()
    plt.tight_layout()

    out = OUTDIR / f"fig_isolated_buffer_delta_BR_H{str(H).replace('.', '_')}.png"
    plt.savefig(out, dpi=300)
    plt.show()

print("Saved isolated buffer-effect outputs to:", OUTDIR.resolve())

In [ ]:
# ==========================================================
# R2 Major Comment 10:
# Severity metric sensitivity:
# UM|BO p75 vs p90 vs p95
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from scipy.stats import spearmanr, t
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

OUTDIR = REVIEW_OUTDIR / "R2_major10_severity_sensitivity"
OUTDIR.mkdir(parents=True, exist_ok=True)

if "trade_master_review" in globals():
    df0_review = trade_master_review.copy()
else:
    df0_review = pd.read_csv(REVIEW_OUTDIR / "trade_master_review.csv")

MAIN_PROTOCOL = "standard"
MAIN_MODEL = "rf"
MAIN_ALPHA = 0.20
MAIN_BUFFER = "fixed"
MAIN_SCENARIO = "strict"

POLICIES_TO_REPORT = [
    "ML fixed (train-selected)",
    "ML adaptive",
]

required_cols = [
    "unmet_minutes_blackout_p75",
    "unmet_minutes_blackout_p90",
    "unmet_minutes_blackout_p95",
]

missing = [c for c in required_cols if c not in df0_review.columns]
if missing:
    raise ValueError(f"Missing required severity columns: {missing}")

df = df0_review.copy()

df = df[
    (df["protocol"] == MAIN_PROTOCOL) &
    (df["model"] == MAIN_MODEL) &
    (df["Scenario"] == MAIN_SCENARIO) &
    (df["Policy"].isin(POLICIES_TO_REPORT)) &
    (df["Budget"].notna()) &
    (df["buffer_mode"] == MAIN_BUFFER) &
    (np.isclose(df["alpha"], MAIN_ALPHA))
].copy()

df["replicate_id"] = (
    df["split_seed"].astype(str) + "_" +
    df["model_seed"].astype(str)
)

print("Rows used:", len(df))
print("Policies:", sorted(df["Policy"].unique()))
print("Replicates:", df["replicate_id"].nunique())

def ci95_series(x):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)

    if n == 0:
        return pd.Series({
            "mean": np.nan,
            "std": np.nan,
            "n": 0,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "ci95_half_width": np.nan,
        })

    mean = float(x.mean())
    std = float(x.std(ddof=1)) if n > 1 else 0.0

    if n > 1:
        crit = float(t.ppf(0.975, df=n - 1)) if SCIPY_AVAILABLE else 1.96
        half_width = crit * std / np.sqrt(n)
    else:
        half_width = 0.0

    return pd.Series({
        "mean": mean,
        "std": std,
        "n": n,
        "ci95_low": mean - half_width,
        "ci95_high": mean + half_width,
        "ci95_half_width": half_width,
    })


# --------------------------
# Severity CI table
# --------------------------
severity_metrics = [
    "unmet_minutes_blackout_p75",
    "unmet_minutes_blackout_p90",
    "unmet_minutes_blackout_p95",
]

rows = []

for keys, g in df.groupby(["Night_h", "Budget", "Policy"], dropna=False):
    H, B, policy = keys

    base = {
        "Night_h": H,
        "Budget": B,
        "Policy": policy,
    }

    for metric in severity_metrics:
        stats = ci95_series(g[metric])

        row = base.copy()
        row["metric"] = metric
        row.update(stats.to_dict())
        rows.append(row)

severity_ci_long = pd.DataFrame(rows)

severity_ci_long.to_csv(
    OUTDIR / "table_R2_major10_UMBO_p75_p90_p95_95CI_long.csv",
    index=False,
)

severity_ci_wide = severity_ci_long.pivot_table(
    index=["Night_h", "Budget", "Policy"],
    columns="metric",
    values=["mean", "ci95_low", "ci95_high", "ci95_half_width", "n"],
    aggfunc="first",
).reset_index()

severity_ci_wide.columns = [
    "_".join([str(c) for c in col if str(c) != ""]).strip("_")
    for col in severity_ci_wide.columns
]

severity_ci_wide.to_csv(
    OUTDIR / "table_R2_major10_UMBO_p75_p90_p95_95CI_wide.csv",
    index=False,
)

display(severity_ci_wide.head(30))


# --------------------------
# Ranking stability
# Lower UM|BO percentile = lower conditional blackout severity.
# --------------------------
rank_rows = []

for metric in severity_metrics:
    tmp = df.copy()
    tmp["severity_metric"] = metric
    tmp["severity_value"] = tmp[metric]

    tmp["rank"] = tmp.groupby(
        ["replicate_id", "Night_h", "Budget", "severity_metric"]
    )["severity_value"].rank(method="average", ascending=True)

    rank_rows.append(
        tmp[
            [
                "replicate_id",
                "Night_h",
                "Budget",
                "Policy",
                "severity_metric",
                "severity_value",
                "rank",
            ]
        ]
    )

rank_df = pd.concat(rank_rows, ignore_index=True)

rank_summary_rows = []

for keys, g in rank_df.groupby(
    ["Night_h", "Budget", "Policy", "severity_metric"],
    dropna=False,
):
    H, B, policy, metric = keys

    stats = ci95_series(g["rank"])

    row = {
        "Night_h": H,
        "Budget": B,
        "Policy": policy,
        "severity_metric": metric,
    }

    row.update(stats.to_dict())
    rank_summary_rows.append(row)

rank_summary = pd.DataFrame(rank_summary_rows)

rank_summary.to_csv(
    OUTDIR / "table_R2_major10_policy_rank_stability_p75_p90_p95.csv",
    index=False,
)

display(rank_summary.head(40))


# --------------------------
# Spearman agreement between p90 and alternatives
# --------------------------
agreement_rows = []

for keys, g in severity_ci_wide.groupby(["Night_h", "Budget"], dropna=False):
    H, B = keys

    c75 = "mean_unmet_minutes_blackout_p75"
    c90 = "mean_unmet_minutes_blackout_p90"
    c95 = "mean_unmet_minutes_blackout_p95"

    if SCIPY_AVAILABLE and len(g) >= 3:
        rho_75_90, p_75_90 = spearmanr(g[c75], g[c90])
        rho_90_95, p_90_95 = spearmanr(g[c90], g[c95])
    else:
        rho_75_90, p_75_90 = np.nan, np.nan
        rho_90_95, p_90_95 = np.nan, np.nan

    agreement_rows.append({
        "Night_h": H,
        "Budget": B,
        "spearman_p75_vs_p90": rho_75_90,
        "pvalue_p75_vs_p90": p_75_90,
        "spearman_p90_vs_p95": rho_90_95,
        "pvalue_p90_vs_p95": p_90_95,
    })

rank_agreement = pd.DataFrame(agreement_rows)

rank_agreement.to_csv(
    OUTDIR / "table_R2_major10_spearman_rank_agreement.csv",
    index=False,
)

display(rank_agreement)


# --------------------------
# Plot p75 / p90 / p95 sensitivity
# --------------------------
PLOT_POLICIES = [
    "ML fixed (train-selected)",
    "ML adaptive",
    "MPC-style baseline",
]

metric_label = {
    "unmet_minutes_blackout_p75": "UM|BO p75",
    "unmet_minutes_blackout_p90": "UM|BO p90",
    "unmet_minutes_blackout_p95": "UM|BO p95",
}

plot_long = severity_ci_long[
    severity_ci_long["Policy"].isin(PLOT_POLICIES)
].copy()

plot_long["metric_label"] = plot_long["metric"].map(metric_label)

for H in sorted(plot_long["Night_h"].unique()):
    for B in sorted(plot_long["Budget"].unique()):

        d = plot_long[
            (np.isclose(plot_long["Night_h"], H)) &
            (np.isclose(plot_long["Budget"], B))
        ].copy()

        if d.empty:
            continue

        labels = []
        x = []
        y = []
        yerr = []

        for i, (_, row) in enumerate(d.iterrows()):
            labels.append(f"{row['Policy']}\n{row['metric_label']}")
            x.append(i)
            y.append(row["mean"])
            yerr.append(row["ci95_half_width"])

        plt.figure(figsize=(10, 5))
        plt.bar(x, y, yerr=yerr, capsize=4)
        plt.xticks(x, labels, rotation=40, ha="right")
        plt.ylabel("UM|BO severity percentile (minutes)")
        plt.title(f"Severity sensitivity: p75, p90, p95, H={H}h, B={B}%")
        plt.tight_layout()

        out = OUTDIR / f"fig_R2_major10_severity_p75_p90_p95_H{str(H).replace('.', '_')}_B{int(B)}.png"
        plt.savefig(out, dpi=300)
        plt.show()

print("Saved R2 Major 10 outputs to:", OUTDIR.resolve())